# ARC AGI 3 — Goose-T1 hybrid (online change-reward CNN + bc_v4 as decaying soft prior)

**Approach (online learning + pretrained prior):**
1. **Small CNN** (16 -> 32 -> 64 -> 128 -> 256) with action head (6 logits) + spatial coord head (64x64 logit map for ACTION6).
2. **bc_v4 pretrained checkpoint** loaded at startup; its action_logits, x_logits, y_logits added as a DECAYING soft prior to Goose's logits.
3. **Decay**: prior_weight=4.0 at step 0, linearly decays to 0 over 500 steps. After decay, Goose runs as pure online change-reward.
4. **Online training every 5 steps** via BCE on (selected_logit, did-frame-change-reward).
5. **Reset model + optimizer per level**. bc_v4 prior re-applied fresh each level.

**Why this design (caveat below):** local A/B on masked-id full-25 games showed T1 at 0.163 mean_score vs Pure Goose's 0.063 (2.6x lift). After-the-fact holdout test (bc_v4 retrained on 20 games, eval on 5 unseen) showed this lift was largely memorization (T1_holdout=0.016 ~ T0_holdout=0.014). Kaggle hidden games are truly unseen, so bc_v4's prior may add ~no value here.

**Realistic Kaggle range:** 0.15-0.25 (vs published Pure Goose 0.25). Submitted as one data point in our 2026-05-11 A/B; the next push (Goose-T2 with GRU memory) is the better-bet generalizer.

**Inputs:** competition framework + `jihangli1121/arc-agi-3-replays-v1` Kaggle Dataset (for the bc_v4 best_action.pth checkpoint at /kaggle/working/best.pth).


In [ ]:
# --- Cell 1: install vendored wheels --- #
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
# --- Cell 2: configure paths --- #
from pathlib import Path

# 1. Competition input — provided by Kaggle, contains framework + wheels.
COMPETITION_INPUT = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3')

# 2. YOUR replays dataset. Kaggle mounts datasets at one of two paths
#    depending on how the kernel was set up — try both, prefer the
#    user-confirmed canonical first.
REPLAY_INPUT_CANDIDATES = [
    Path('/kaggle/input/datasets/jihangli1121/arc-agi-3-replays-v1'),  # canonical (user-confirmed 2026-05-10)
    Path('/kaggle/input/arc-agi-3-replays-v1'),                         # slug-only fallback
]
REPLAY_INPUT = next((p for p in REPLAY_INPUT_CANDIDATES if p.exists()), REPLAY_INPUT_CANDIDATES[0])

WORK = Path('/kaggle/working')
REPLAY_BASE_DIR = WORK / 'replays'
REPLAY_BASE_DIR.mkdir(exist_ok=True)

assert COMPETITION_INPUT.exists(), f'Competition input missing: {COMPETITION_INPUT}'
print(f'COMPETITION_INPUT: {COMPETITION_INPUT}  (exists)')
print(f'REPLAY_INPUT:      {REPLAY_INPUT}  (exists: {REPLAY_INPUT.exists()})')
if not REPLAY_INPUT.exists():
    print(f'  candidates tried: {[str(p) for p in REPLAY_INPUT_CANDIDATES]}')
    print('  WARNING: agent will run fully random (no replays).')
print('paths OK')

In [ ]:
# --- Cell 3: stage replay files into REPLAY_BASE_DIR --- #
import os

if REPLAY_INPUT.exists():
    src_root = REPLAY_INPUT / 'environment_files'
    if not src_root.is_dir():
        src_root = REPLAY_INPUT  # tolerate flat layout
    n_games_with_replays = 0
    for game_dir in sorted(p for p in src_root.iterdir() if p.is_dir()):
        replays_src = game_dir / 'replays'
        if not replays_src.is_dir():
            continue
        replays_dst = REPLAY_BASE_DIR / game_dir.name / 'replays'
        replays_dst.parent.mkdir(parents=True, exist_ok=True)
        if replays_dst.exists() or replays_dst.is_symlink():
            continue
        os.symlink(replays_src, replays_dst)
        n_games_with_replays += 1
    print(f'staged replays for {n_games_with_replays} games into {REPLAY_BASE_DIR}')
else:
    print('no REPLAY_INPUT — agent will run with no replays (random fallback)')

os.environ['ARC_REPLAY_BASE_DIR'] = str(REPLAY_BASE_DIR)

# --- v4.3 TTT: stage the BC checkpoint where my_agent expects it ---
import shutil as _sh
for src_path in [
    '/kaggle/input/arc-agi-3-replays-v1/best.pth',
    '/kaggle/input/datasets/jihangli1121/arc-agi-3-replays-v1/best.pth',
]:
    p = Path(src_path)
    if p.is_file():
        _sh.copy(str(p), '/kaggle/working/best.pth')
        print(f'[stage] copied BC checkpoint {p} -> /kaggle/working/best.pth ({p.stat().st_size/1e6:.1f} MB)')
        break
else:
    print('[stage] no BC checkpoint found in dataset — TTT will be disabled')


In [ ]:
%%writefile /kaggle/working/my_agent.py
# =====================================================================
# GooseAgent — online change-reward CNN policy
#
# Re-implementation of the StochasticGoose pattern from the public ARC
# AGI 3 sample submission (Tufa Labs, Smit + Cole). Same core idea:
#   - small CNN encodes 16-color one-hot 64x64 frame
#   - action head (6 logits for ACTION{1,2,3,4,5,7}) + coord head (64x64 = 4096 logits for ACTION6)
#   - intrinsic reward: "did the frame change after my action?"
#   - online BCE training every train_frequency steps on (selected_logit, reward)
#   - reset model + optimizer at every new level
#
# Differences vs the public sample:
#   - 7 game actions, not 5 (the sample dropped ACTION7).
#   - Pluggable deltas via env vars so we can run T1..T4 experiments
#     against the T0 anchor without forking the file.
#   - Same MyAgent base class our framework uses (compatibility with
#     scripts/test_my_agent_local.py and the Kaggle gateway runner).
# =====================================================================
from __future__ import annotations

import hashlib
import os
import random
import time
import traceback
from collections import deque
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent


# Action IDs that are NOT click (ACTION6 is the click). We treat these as
# the simple action head's 6 outputs, in this fixed order so action_idx
# maps to an action id deterministically.
SIMPLE_ACTION_IDS: Tuple[int, ...] = (1, 2, 3, 4, 5, 7)
NUM_SIMPLE_ACTIONS = len(SIMPLE_ACTION_IDS)
GRID_SIZE = 64
NUM_COLORS = 16
NUM_COORDS = GRID_SIZE * GRID_SIZE


def _env_flag(name: str, default: bool = False) -> bool:
    return str(os.environ.get(name, '1' if default else '')).strip() in ('1', 'true', 'TRUE', 'yes')


def _env_float(name: str, default: float) -> float:
    try:
        return float(os.environ.get(name, default))
    except (TypeError, ValueError):
        return default


def _available_action_ids(latest_frame: FrameData) -> List[int]:
    raw = getattr(latest_frame, 'available_actions', None)
    if raw is None:
        return list(range(1, 8))
    out = []
    for a in raw:
        try:
            v = a.value if hasattr(a, 'value') else int(a)
            if 1 <= int(v) <= 7:
                out.append(int(v))
        except Exception:
            continue
    return out or list(range(1, 8))


def _final_subframe(frame_data: FrameData) -> np.ndarray:
    """Pull the last 64x64 grid from FrameData.frame (list of stacked grids)."""
    arr = np.asarray(frame_data.frame, dtype=np.int64)
    if arr.ndim == 3:
        arr = arr[-1]
    return arr


class ActionModel(nn.Module):
    """Plain 4-conv CNN with action + spatial coord heads.

    Identical conv shape family to the published StochasticGoose model
    (16 -> 32 -> 64 -> 128 -> 256, stride 1 everywhere). Coord head is a
    fully-convolutional spatial decoder that outputs a 64x64 logit map.
    """

    def __init__(
        self,
        input_channels: int = NUM_COLORS,
        grid_size: int = GRID_SIZE,
        num_simple_actions: int = NUM_SIMPLE_ACTIONS,
    ) -> None:
        super().__init__()
        self.grid_size = grid_size
        self.num_simple_actions = num_simple_actions

        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)

        # Simple-action head
        self.action_pool = nn.MaxPool2d(4, 4)
        action_flat = 256 * (grid_size // 4) * (grid_size // 4)
        self.action_fc = nn.Linear(action_flat, 512)
        self.action_head = nn.Linear(512, num_simple_actions)

        # Coord head (64x64 spatial logits for ACTION6)
        self.coord_conv1 = nn.Conv2d(256, 128, kernel_size=3, padding=1)
        self.coord_conv2 = nn.Conv2d(128, 64, kernel_size=3, padding=1)
        self.coord_conv3 = nn.Conv2d(64, 32, kernel_size=1)
        self.coord_conv4 = nn.Conv2d(32, 1, kernel_size=1)

        self.dropout = nn.Dropout(0.2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return concatenated logits: [simple_actions (6), coord_flat (4096)] = (B, 4102)."""
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        feats = F.relu(self.conv4(x))  # (B, 256, 64, 64)

        a = self.action_pool(feats)
        a = a.view(a.size(0), -1)
        a = F.relu(self.action_fc(a))
        a = self.dropout(a)
        action_logits = self.action_head(a)  # (B, 6)

        c = F.relu(self.coord_conv1(feats))
        c = F.relu(self.coord_conv2(c))
        c = F.relu(self.coord_conv3(c))
        coord_logits = self.coord_conv4(c).view(c.size(0), -1)  # (B, 4096)

        return torch.cat([action_logits, coord_logits], dim=1)


class MyAgent(Agent):
    """Online change-reward CNN policy. Resets per level.

    Pluggable deltas via env vars:
      ARC_GOOSE_DELTA   = 'none' | 't1_bc' | 't2_gru' | 't3_priors' | 't4_goal'
      ARC_GOOSE_LR      = AdamW lr (default 1e-4)
      ARC_GOOSE_TRAIN_EVERY = train_frequency (default 5)
      ARC_GOOSE_BUFFER  = experience buffer maxlen (default 200000)
      ARC_GOOSE_BATCH   = train batch size (default 64)
    """

    MAX_ACTIONS = float('inf')
    _MAX_FRAMES = 10

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        seed = (int(time.time() * 1_000_000) + hash(self.game_id)) & 0xFFFFFFFF
        random.seed(seed)
        np.random.seed(seed % (2**32 - 1))
        torch.manual_seed(seed % (2**32 - 1))
        self._rng = random.Random(seed)

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.delta = os.environ.get('ARC_GOOSE_DELTA', 'none').strip().lower()
        self.lr = _env_float('ARC_GOOSE_LR', 1e-4)
        self.train_every = int(_env_float('ARC_GOOSE_TRAIN_EVERY', 5))
        self.buffer_maxlen = int(_env_float('ARC_GOOSE_BUFFER', 200000))
        self.batch_size = int(_env_float('ARC_GOOSE_BATCH', 64))

        self.short_id = self.game_id.split('-', 1)[0] if self.game_id else ''
        self.start_time = time.time()
        self.current_level = -1
        self.action_model: Optional[ActionModel] = None
        self.optimizer: Optional[optim.Optimizer] = None
        self.experience_buffer: deque = deque(maxlen=self.buffer_maxlen)
        self.experience_hashes: set = set()
        self.prev_frame: Optional[np.ndarray] = None  # bool (16, 64, 64)
        self.prev_action_idx: Optional[int] = None
        self.online_step_count = 0

        # T1/T3 prior decay knobs (shared)
        self._prior_decay_steps = int(_env_float('ARC_GOOSE_PRIOR_DECAY', 500))
        self._prior_weight_init = _env_float('ARC_GOOSE_PRIOR_WEIGHT', 4.0)

        # T1 delta state (bc_v4 logits as decaying prior)
        self._bc_helper = None
        self._bc_history: deque = deque(maxlen=4)
        self._bc_last_action_id: Optional[int] = None
        self._bc_step_index: int = 0
        self._bc_steps_since_progress: int = 0
        self._bc_levels_completed: int = 0
        # ARC_T1_PERM_BC_COLORS=1 permutes the 16-color palette before
        # feeding frames to bc_v4 — a memorization-vs-generalization test.
        # If bc_v4 learned structural patterns, perm-on ~= perm-off.
        # If bc_v4 memorized specific color sprites, perm-on collapses
        # toward T0 (Pure Goose).
        self._bc_color_perm_enabled = (
            str(os.environ.get('ARC_T1_PERM_BC_COLORS', '')).strip() in ('1', 'true', 'TRUE', 'yes')
        )
        self._bc_color_perm: Optional[np.ndarray] = None  # length-16, set per level
        if self.delta == 't1_bc':
            try:
                from bc_policy import PolicyHelper, find_checkpoint
                from pathlib import Path as _Path
                ckpt_env = os.environ.get('ARC_BC_CHECKPOINT_PATH')
                ckpt_path = _Path(ckpt_env) if ckpt_env else find_checkpoint()
                if ckpt_path is None:
                    print('[GooseAgent] T1: no bc_v4 checkpoint found; falling back to T0', flush=True)
                else:
                    self._bc_helper = PolicyHelper.load(ckpt_path, device=self.device)
            except Exception as ex:
                print(f'[GooseAgent] T1 bc_v4 load failed: {ex}', flush=True)
                self._bc_helper = None
            print(
                f'[GooseAgent] T1 bc prior: helper_loaded={self._bc_helper is not None} '
                f'decay_steps={self._prior_decay_steps} init_weight={self._prior_weight_init}',
                flush=True,
            )

        # T3 delta state (lazy load to keep T0 path clean)
        self._effect_dict = None
        if self.delta == 't3_priors':
            try:
                raise ImportError('T3 path disabled on Kaggle (would self-import)')
                self._effect_dict = _load_action_effect_dict()
                self._extract_saliency = _extract_saliency
            except Exception as ex:
                print(f'[GooseAgent] T3 prior load failed: {ex}', flush=True)
                self._extract_saliency = None
            print(
                f'[GooseAgent] T3 priors: dict_loaded={self._effect_dict is not None} '
                f'decay_steps={self._prior_decay_steps} init_weight={self._prior_weight_init}',
                flush=True,
            )

        print(
            f'[GooseAgent] init game_id={self.game_id} short_id={self.short_id} '
            f'delta={self.delta} device={self.device} lr={self.lr} '
            f'train_every={self.train_every}',
            flush=True,
        )

    def append_frame(self, frame: FrameData) -> None:
        self.frames.append(frame)
        if len(self.frames) > self._MAX_FRAMES:
            self.frames = self.frames[-self._MAX_FRAMES:]
        if frame.guid:
            self.guid = frame.guid
        if hasattr(self, 'recorder') and not getattr(self, 'is_playback', False):
            import json
            try:
                self.recorder.record(json.loads(frame.model_dump_json()))
            except Exception:
                pass

    def is_done(self, frames: List[FrameData], latest_frame: FrameData) -> bool:
        try:
            return latest_frame.state is GameState.WIN
        except Exception:
            return False

    def _frame_to_tensor(self, frame_data: FrameData) -> torch.Tensor:
        frame = _final_subframe(frame_data)
        if frame.shape != (GRID_SIZE, GRID_SIZE):
            raise ValueError(f'unexpected frame shape {frame.shape}')
        t = torch.zeros(NUM_COLORS, GRID_SIZE, GRID_SIZE, dtype=torch.float32, device=self.device)
        idx = torch.from_numpy(frame).long().clamp(0, NUM_COLORS - 1).to(self.device)
        t.scatter_(0, idx.unsqueeze(0), 1)
        return t

    def _build_model_for_new_level(self) -> None:
        self.experience_buffer.clear()
        self.experience_hashes.clear()
        self.action_model = ActionModel().to(self.device)
        self.optimizer = optim.Adam(self.action_model.parameters(), lr=self.lr)
        self.prev_frame = None
        self.prev_action_idx = None
        self.online_step_count = 0

    def _experience_hash(self, frame_bool: np.ndarray, action_idx: int) -> str:
        return hashlib.md5(frame_bool.tobytes() + str(action_idx).encode()).hexdigest()

    def _train(self) -> None:
        if self.action_model is None or self.optimizer is None:
            return
        if len(self.experience_buffer) < self.batch_size:
            return
        idxs = np.random.choice(len(self.experience_buffer), self.batch_size, replace=False)
        batch = [self.experience_buffer[i] for i in idxs]
        states = torch.stack([torch.from_numpy(e['state']).float().to(self.device) for e in batch])
        action_idx = torch.tensor([e['action_idx'] for e in batch], dtype=torch.long, device=self.device)
        rewards = torch.tensor([e['reward'] for e in batch], dtype=torch.float32, device=self.device)

        self.optimizer.zero_grad()
        logits = self.action_model(states)
        selected = logits.gather(1, action_idx.unsqueeze(1)).squeeze(1)
        loss = F.binary_cross_entropy_with_logits(selected, rewards)
        # Small entropy bonus on the full distribution to keep coords exploring.
        with torch.no_grad():
            probs = torch.sigmoid(logits)
        entropy = -(probs[:, :NUM_SIMPLE_ACTIONS].mean() * 1e-4
                    + probs[:, NUM_SIMPLE_ACTIONS:].mean() * 1e-5)
        (loss + entropy).backward()
        self.optimizer.step()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def _build_priors(
        self,
        latest_frame: FrameData,
    ) -> Tuple[Optional[torch.Tensor], Optional[torch.Tensor]]:
        """Compute (simple_prior_6, coord_prior_4096) tensors per the active delta.

        Each returned tensor is on self.device, or None if no prior applies.
        Both T1 and T3 priors decay linearly with self.online_step_count.
        """
        simple_prior: Optional[torch.Tensor] = None
        coord_prior: Optional[torch.Tensor] = None

        # ---- T3: salient + effect-dict prior ----
        if self.delta == 't3_priors' and self._extract_saliency is not None:
            try:
                raw = _final_subframe(latest_frame).tolist()
            except Exception:
                raw = None
            if raw is not None:
                salient = self._extract_saliency(raw)
                cp = torch.zeros(NUM_COORDS, dtype=torch.float32, device=self.device)
                for (x, y) in salient[:30]:
                    if 0 <= x < GRID_SIZE and 0 <= y < GRID_SIZE:
                        cp[y * GRID_SIZE + x] += 1.0
                if self._effect_dict is not None:
                    try:
                        arr = np.asarray(raw, dtype=np.int64).reshape(-1)
                        hist = np.bincount(arr, minlength=NUM_COLORS).astype(np.float32)
                        norm = np.linalg.norm(hist) + 1e-6
                        qv = hist / norm
                        target = self._effect_dict.feature_keys.shape[1]
                        if qv.shape[0] != target:
                            if qv.shape[0] < target:
                                qv = np.pad(qv, (0, target - qv.shape[0]))
                            else:
                                qv = qv[:target]
                        for (x, y) in self._effect_dict.top_clicks_by_similarity(qv, k=10):
                            if 0 <= x < GRID_SIZE and 0 <= y < GRID_SIZE:
                                cp[y * GRID_SIZE + x] += 0.5
                    except Exception as ex:
                        if self.online_step_count == 0:
                            print(f'[GooseAgent] T3 dict query failed: {ex}', flush=True)
                coord_prior = cp

        # ---- T1: bc_v4 logits as prior ----
        if self.delta == 't1_bc' and self._bc_helper is not None:
            try:
                raw_arr = _final_subframe(latest_frame)
                # Memorization test: optionally permute colors before feeding to bc_v4.
                # x_logits / y_logits are coordinate priors and unaffected by color
                # permutation in semantics — only the visual frame content shifts.
                if self._bc_color_perm is not None:
                    raw_for_bc = self._bc_color_perm[raw_arr].tolist()
                else:
                    raw_for_bc = raw_arr.tolist()
                self._bc_history.append(raw_for_bc)
                scores = self._bc_helper.score_frame(
                    history_frames=list(self._bc_history),
                    latest_frame=raw_for_bc,
                    last_action_id=self._bc_last_action_id,
                    levels_completed=self._bc_levels_completed,
                    steps_since_progress=self._bc_steps_since_progress,
                    step_index=self._bc_step_index,
                    available_actions=_available_action_ids(latest_frame),
                )
                if scores is not None:
                    a = scores['action_logits']  # (7,) for ACTION1..ACTION7
                    x = scores['x_logits']        # (64,)
                    y = scores['y_logits']        # (64,)
                    sp = torch.zeros(NUM_SIMPLE_ACTIONS, dtype=torch.float32, device=self.device)
                    for i, aid in enumerate(SIMPLE_ACTION_IDS):
                        sp[i] = float(a[aid - 1])
                    # outer-product of y and x logits → 64x64 → flat 4096
                    yy = torch.from_numpy(y.astype(np.float32)).to(self.device)
                    xx = torch.from_numpy(x.astype(np.float32)).to(self.device)
                    cp = (yy.unsqueeze(1) + xx.unsqueeze(0)).reshape(-1)
                    simple_prior = sp
                    coord_prior = cp
            except Exception as ex:
                if self.online_step_count == 0:
                    print(f'[GooseAgent] T1 bc score failed: {ex}', flush=True)
            self._bc_step_index += 1

        return simple_prior, coord_prior

    def _sample_action(
        self,
        combined_logits: torch.Tensor,
        available_ids: List[int],
        simple_prior: Optional[torch.Tensor] = None,
        coord_prior: Optional[torch.Tensor] = None,
    ) -> Tuple[int, Optional[Tuple[int, int]], int]:
        """Return (action_idx, coords_or_none, full_idx) where full_idx indexes the combined output."""
        simple_logits = combined_logits[:NUM_SIMPLE_ACTIONS].clone()
        coord_logits = combined_logits[NUM_SIMPLE_ACTIONS:].clone()

        # Apply decaying priors to logits.
        decay = max(0.0, 1.0 - self.online_step_count / max(1, self._prior_decay_steps))
        if simple_prior is not None:
            simple_logits = simple_logits + self._prior_weight_init * decay * simple_prior
        if coord_prior is not None:
            coord_logits = coord_logits + self._prior_weight_init * decay * coord_prior

        action6_available = 6 in available_ids
        simple_available_mask = torch.tensor(
            [1.0 if aid in available_ids else 0.0 for aid in SIMPLE_ACTION_IDS],
            dtype=torch.float32,
            device=combined_logits.device,
        )
        simple_logits = torch.where(
            simple_available_mask > 0,
            simple_logits,
            torch.full_like(simple_logits, float('-inf')),
        )
        if not action6_available:
            coord_logits = torch.full_like(coord_logits, float('-inf'))

        simple_probs = torch.sigmoid(simple_logits)
        coord_probs = torch.sigmoid(coord_logits) / NUM_COORDS

        all_probs = torch.cat([simple_probs, coord_probs])
        total = all_probs.sum().item()
        if not np.isfinite(total) or total <= 0:
            # Last-resort uniform over the available actions
            uniform = torch.zeros_like(all_probs)
            for aid in available_ids:
                if aid == 6:
                    uniform[NUM_SIMPLE_ACTIONS:] = 1.0 / NUM_COORDS
                elif aid in SIMPLE_ACTION_IDS:
                    uniform[SIMPLE_ACTION_IDS.index(aid)] = 1.0
            all_probs = uniform
            total = all_probs.sum().item()
        all_probs = all_probs / total
        probs_np = all_probs.detach().cpu().numpy()
        full_idx = int(np.random.choice(len(probs_np), p=probs_np))

        if full_idx < NUM_SIMPLE_ACTIONS:
            return full_idx, None, full_idx
        coord_idx = full_idx - NUM_SIMPLE_ACTIONS
        y = coord_idx // GRID_SIZE
        x = coord_idx % GRID_SIZE
        return NUM_SIMPLE_ACTIONS, (int(y), int(x)), full_idx

    def choose_action(self, frames: List[FrameData], latest_frame: FrameData) -> GameAction:
        try:
            # Reset model when level changes
            new_level = int(getattr(latest_frame, 'levels_completed', 0))
            if new_level != self.current_level:
                print(f'[GooseAgent] level change {self.current_level}->{new_level}; resetting model', flush=True)
                self._build_model_for_new_level()
                self.current_level = new_level
                # Reset T1 BC state on level change too (bc_v4 history becomes stale).
                self._bc_history.clear()
                self._bc_last_action_id = None
                self._bc_step_index = 0
                self._bc_steps_since_progress = 0
                self._bc_levels_completed = new_level
                # T1 memorization-test: fresh color permutation per level.
                if self._bc_color_perm_enabled:
                    self._bc_color_perm = np.arange(NUM_COLORS)
                    np.random.shuffle(self._bc_color_perm)
                    print(f'[GooseAgent] T1 perm: {self._bc_color_perm.tolist()}', flush=True)
                else:
                    self._bc_color_perm = None
            else:
                self._bc_steps_since_progress += 1

            # Hard reset of episode if RESET state
            state = getattr(latest_frame, 'state', None)
            if state is GameState.NOT_PLAYED or state is GameState.GAME_OVER:
                self.prev_frame = None
                self.prev_action_idx = None
                action = GameAction.RESET
                action.reasoning = {'strategy': 'goose', 'phase': 'reset'}
                return action

            current = self._frame_to_tensor(latest_frame)
            current_bool = current.detach().cpu().numpy().astype(bool)

            # Record experience from previous step: did the frame change?
            if self.prev_frame is not None and self.prev_action_idx is not None:
                exp_hash = self._experience_hash(self.prev_frame, self.prev_action_idx)
                if exp_hash not in self.experience_hashes:
                    changed = not np.array_equal(self.prev_frame, current_bool)
                    self.experience_buffer.append({
                        'state': self.prev_frame,
                        'action_idx': self.prev_action_idx,
                        'reward': 1.0 if changed else 0.0,
                    })
                    self.experience_hashes.add(exp_hash)

            available_ids = _available_action_ids(latest_frame)
            with torch.no_grad():
                logits = self.action_model(current.unsqueeze(0)).squeeze(0)

            simple_prior, coord_prior = self._build_priors(latest_frame)
            action_slot, coords, full_idx = self._sample_action(
                logits, available_ids, simple_prior, coord_prior
            )

            if action_slot < NUM_SIMPLE_ACTIONS:
                aid = SIMPLE_ACTION_IDS[action_slot]
                act = GameAction.from_id(int(aid))
                act.reasoning = {'strategy': 'goose', 'phase': 'simple', 'aid': aid}
                self._bc_last_action_id = int(aid)
            else:
                y, x = coords
                act = GameAction.ACTION6
                act.set_data({'x': int(x), 'y': int(y)})
                act.reasoning = {'strategy': 'goose', 'phase': 'click', 'xy': (int(x), int(y))}
                self._bc_last_action_id = 6

            self.prev_frame = current_bool
            self.prev_action_idx = full_idx
            self.online_step_count += 1

            if self.online_step_count % self.train_every == 0:
                self._train()

            return act

        except Exception as e:
            print(f'[GooseAgent] choose_action crashed: {type(e).__name__}: {e}', flush=True)
            traceback.print_exc()
            # Fallback: a safe random simple action
            avail = _available_action_ids(latest_frame)
            simple_avail = [a for a in avail if a in SIMPLE_ACTION_IDS]
            aid = self._rng.choice(simple_avail) if simple_avail else 1
            act = GameAction.from_id(int(aid))
            act.reasoning = {'strategy': 'goose', 'phase': 'fallback_error', 'err': str(e)}
            return act


In [ ]:
%%writefile /kaggle/working/bc_policy.py
"""Vendored BC policy + TTT primitives for the Kaggle submission notebook.

Self-contained — imports only torch, numpy, json, pathlib. Mirrors the parts of
src/model.py + src/common.py that PolicyGuidedAgent uses, without dragging in
the rest of the project (collect/evaluate/etc).

Two distinct things in here:

1. **ObjectCentricPolicy** — exact copy of `src/model.py:ObjectCentricPolicy`.
   Loads a checkpoint trained by `python -m src.train` and produces per-frame
   `action_logits / x_logits / y_logits / value / avail_logits` scores.

2. **TTT (test-time training)** — adapts the loaded checkpoint to the SPECIFIC
   game being played, at submission time, on Kaggle's GPU. Two modes:
     - `replay_finetune` (public games): SGD on this game's GT replay
     - `aug_finetune`    (any game): color-permutation consistency loss

Local testing: scripts/test_my_agent_local.py imports my_agent.py which now
imports bc_policy. Run with ARC_BC_CHECKPOINT_PATH pointing at any .pth file.
Disable via ARC_DISABLE_TTT=1 to A/B without TTT.
"""
from __future__ import annotations

import json
import os
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    _TORCH_OK = True
except Exception:
    torch = None  # type: ignore
    nn = None  # type: ignore
    F = None  # type: ignore
    _TORCH_OK = False


GRID_SIZE = 64
NUM_COLORS = 16
ACTION_IDS: Tuple[int, ...] = (1, 2, 3, 4, 5, 6, 7)
ACTION_TO_INDEX: Dict[int, int] = {aid: i for i, aid in enumerate(ACTION_IDS)}


# ---------------------- frame / scalar helpers (vendored from src/common.py) ----------------------


def _safe_color(value: Any) -> int:
    try:
        v = int(value)
    except (TypeError, ValueError):
        return 0
    if v < 0:
        return 0
    if v >= NUM_COLORS:
        return v % NUM_COLORS
    return v


def _is_iterable(obj: Any) -> bool:
    try:
        iter(obj)
    except TypeError:
        return False
    return True


def _action_mask(available_actions: Sequence[int]) -> List[float]:
    mask: List[float] = []
    avail_set = {int(a) for a in available_actions}
    for aid in ACTION_IDS:
        mask.append(1.0 if aid in avail_set else 0.0)
    return mask


def _non_background_density(frame: Sequence[Sequence[int]]) -> float:
    total = 0
    nonbg = 0
    for row in frame:
        if not _is_iterable(row):
            continue
        for c in row:
            total += 1
            try:
                if int(c) != 0:
                    nonbg += 1
            except (TypeError, ValueError):
                continue
    return float(nonbg) / max(1, total)


def pad_history(frames: Sequence[Sequence[Sequence[int]]], history: int) -> List[List[List[int]]]:
    """Pad a frame history to `history` frames. Repeats the oldest frame if short."""
    if not frames:
        blank = [[0 for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]
        return [blank for _ in range(history)]
    stacked: List[List[List[int]]] = []
    for frame in frames[-history:]:
        rows: List[List[int]] = []
        for row in frame:
            if _is_iterable(row):
                rows.append([_safe_color(v) for v in row])
            else:
                rows.append([0 for _ in range(GRID_SIZE)])
        stacked.append(rows)
    while len(stacked) < history:
        stacked.insert(0, [row[:] for row in stacked[0]])
    return stacked


def scalar_features(
    available_actions: Sequence[int],
    last_action_id: Optional[int],
    levels_completed: int,
    steps_since_progress: int,
    step_index: int,
    frame: Sequence[Sequence[int]],
    max_steps: int,
) -> "torch.Tensor":
    """18-dim scalar feature vector: 7 action mask + 7 last-action one-hot + 4 progress floats."""
    features: List[float] = []
    features.extend(_action_mask(available_actions))
    last_action = [0.0 for _ in ACTION_IDS]
    if last_action_id in ACTION_TO_INDEX:
        last_action[ACTION_TO_INDEX[last_action_id]] = 1.0
    features.extend(last_action)
    features.extend([
        min(levels_completed / 10.0, 1.0),
        min(steps_since_progress / max(max_steps, 1), 1.0),
        min(step_index / max(max_steps, 1), 1.0),
        _non_background_density(frame),
    ])
    return torch.tensor(features, dtype=torch.float32)


def frames_to_obs_uint8(history_frames: Sequence[Sequence[Sequence[int]]], history: int) -> "torch.Tensor":
    """Build (history, 64, 64) uint8 tensor for the model's GPU one_hot path."""
    padded = pad_history(history_frames, history)
    arr = np.asarray(padded, dtype=np.uint8)
    # Defensive shape correction.
    if arr.shape != (history, GRID_SIZE, GRID_SIZE):
        out = np.zeros((history, GRID_SIZE, GRID_SIZE), dtype=np.uint8)
        h = min(arr.shape[0], history) if arr.ndim >= 1 else 0
        ry = min(arr.shape[1], GRID_SIZE) if arr.ndim >= 2 else 0
        rx = min(arr.shape[2], GRID_SIZE) if arr.ndim >= 3 else 0
        if h and ry and rx:
            out[:h, :ry, :rx] = arr[:h, :ry, :rx]
        arr = out
    return torch.from_numpy(arr & 0x0F)


# ---------------------- ObjectCentricPolicy (verbatim from src/model.py) ----------------------


if _TORCH_OK:
    class ObjectCentricPolicy(nn.Module):
        def __init__(
            self,
            history: int = 4,
            model_dim: int = 384,
            num_slots: int = 8,
            depth: int = 6,
            num_heads: int = 8,
            scalar_dim: int = 18,
            use_goal: bool = False,
        ) -> None:
            super().__init__()
            input_channels = history * 16
            hidden = max(model_dim // 2, 128)
            self.history = history
            self.model_dim = model_dim
            self.num_slots = num_slots
            self.use_goal = bool(use_goal)
            self.goal_dim = 128

            self.conv = nn.Sequential(
                nn.Conv2d(input_channels, hidden // 2, kernel_size=3, padding=1),
                nn.GELU(),
                nn.Conv2d(hidden // 2, hidden, kernel_size=3, stride=2, padding=1),
                nn.GELU(),
                nn.Conv2d(hidden, model_dim, kernel_size=3, stride=2, padding=1),
                nn.GELU(),
                nn.Conv2d(model_dim, model_dim, kernel_size=3, padding=1),
                nn.GELU(),
            )
            self.pos_embed = nn.Parameter(torch.randn(1, 16 * 16, model_dim) * 0.02)
            self.slot_queries = nn.Parameter(torch.randn(1, num_slots, model_dim) * 0.02)
            self.cross_attn = nn.MultiheadAttention(
                embed_dim=model_dim, num_heads=num_heads, batch_first=True,
            )
            self.scalar_proj = nn.Sequential(
                nn.Linear(scalar_dim, model_dim),
                nn.GELU(),
                nn.Linear(model_dim, model_dim),
            )
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=model_dim, nhead=num_heads, dim_feedforward=model_dim * 4,
                activation="gelu", batch_first=True, norm_first=True, dropout=0.1,
            )
            self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
            self.state_norm = nn.LayerNorm(model_dim)

            # Gen 1 goal encoder (mirrors src/model.py). Always constructed for
            # state_dict shape consistency; only USED when self.use_goal is True.
            goal_hidden = 64
            self.goal_encoder = nn.Sequential(
                nn.Conv2d(NUM_COLORS, goal_hidden, kernel_size=3, stride=2, padding=1),
                nn.GELU(),
                nn.Conv2d(goal_hidden, goal_hidden, kernel_size=3, stride=2, padding=1),
                nn.GELU(),
                nn.Conv2d(goal_hidden, self.goal_dim, kernel_size=3, stride=2, padding=1),
                nn.GELU(),
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
            )
            self.goal_proj = nn.Sequential(
                nn.Linear(self.goal_dim, model_dim),
                nn.GELU(),
                nn.Linear(model_dim, model_dim),
            )

            self.action_head = nn.Linear(model_dim, len(ACTION_IDS))
            self.x_head = nn.Linear(model_dim, 64)
            self.y_head = nn.Linear(model_dim, 64)
            self.value_head = nn.Linear(model_dim, 1)
            self.avail_head = nn.Linear(model_dim, len(ACTION_IDS))
            self.action_embed = nn.Embedding(len(ACTION_IDS), model_dim)
            self.next_latent_head = nn.Sequential(
                nn.Linear(model_dim * 2, model_dim),
                nn.GELU(),
                nn.Linear(model_dim, model_dim),
            )

            # Aux heads (off-path at inference; needed only to load checkpoints with these params).
            self.archetype_head = nn.Linear(model_dim, 3)
            saliency_hidden = max(model_dim // 4, 32)
            self._saliency_hidden = saliency_hidden
            self.saliency_decoder = nn.Sequential(
                nn.Conv2d(model_dim, saliency_hidden, kernel_size=3, padding=1),
                nn.GELU(),
                nn.ConvTranspose2d(saliency_hidden, saliency_hidden, kernel_size=4, stride=2, padding=1),
                nn.GELU(),
                nn.ConvTranspose2d(saliency_hidden, 1, kernel_size=4, stride=2, padding=1),
            )
            recon_hidden = max(model_dim // 4, 32)
            self._recon_hidden = recon_hidden
            self.recon_init = nn.Linear(model_dim, recon_hidden * 4 * 4)
            recon_mid = max(recon_hidden // 2, 16)
            self.recon_decoder = nn.Sequential(
                nn.ConvTranspose2d(recon_hidden, recon_hidden, kernel_size=4, stride=2, padding=1),
                nn.GELU(),
                nn.ConvTranspose2d(recon_hidden, recon_hidden, kernel_size=4, stride=2, padding=1),
                nn.GELU(),
                nn.ConvTranspose2d(recon_hidden, recon_mid, kernel_size=4, stride=2, padding=1),
                nn.GELU(),
                nn.ConvTranspose2d(recon_mid, NUM_COLORS, kernel_size=4, stride=2, padding=1),
            )

        def encode_state(
            self,
            obs: "torch.Tensor",
            scalar: "torch.Tensor",
            goal_obs: Optional["torch.Tensor"] = None,
        ) -> Dict[str, "torch.Tensor"]:
            if obs.dim() == 4 and obs.shape[1] == self.history:
                clipped = obs.to(dtype=torch.long).clamp_(0, NUM_COLORS - 1)
                one_hot = F.one_hot(clipped, num_classes=NUM_COLORS)
                obs = one_hot.permute(0, 1, 4, 2, 3).reshape(
                    obs.shape[0], self.history * NUM_COLORS, 64, 64,
                ).to(dtype=torch.float32)
            elif obs.dtype != torch.float32:
                obs = obs.to(dtype=torch.float32)
            features = self.conv(obs)
            batch_size = features.shape[0]
            patches = features.flatten(2).transpose(1, 2)
            patches = patches + self.pos_embed[:, : patches.shape[1], :]
            slots = self.slot_queries.expand(batch_size, -1, -1)
            slot_tokens, _ = self.cross_attn(query=slots, key=patches, value=patches)
            scalar_token = self.scalar_proj(scalar).unsqueeze(1)
            # Gen 1 goal-conditioning (mirrors src/model.py).
            if self.use_goal and goal_obs is not None:
                if goal_obs.dim() == 3:
                    gclip = goal_obs.to(dtype=torch.long).clamp_(0, NUM_COLORS - 1)
                    goal_oh = F.one_hot(gclip, num_classes=NUM_COLORS).permute(0, 3, 1, 2).to(dtype=torch.float32)
                elif goal_obs.dim() == 4 and goal_obs.shape[1] == NUM_COLORS:
                    goal_oh = goal_obs.to(dtype=torch.float32)
                else:
                    goal_oh = None
                if goal_oh is not None:
                    goal_emb = self.goal_encoder(goal_oh)
                    goal_token = self.goal_proj(goal_emb).unsqueeze(1)
                    scalar_token = scalar_token + goal_token
            tokens = torch.cat([scalar_token, slot_tokens], dim=1)
            tokens = self.encoder(tokens)
            pooled = self.state_norm(tokens[:, 0, :])
            return {"pooled": pooled, "tokens": tokens, "patches": patches}

        def forward(
            self,
            obs: "torch.Tensor",
            scalar: "torch.Tensor",
            action_index: Optional["torch.Tensor"] = None,
            goal_obs: Optional["torch.Tensor"] = None,
        ) -> Dict[str, "torch.Tensor"]:
            encoded = self.encode_state(obs, scalar, goal_obs=goal_obs)
            pooled = encoded["pooled"]
            out: Dict[str, "torch.Tensor"] = {
                "pooled": pooled,
                "action_logits": self.action_head(pooled),
                "x_logits": self.x_head(pooled),
                "y_logits": self.y_head(pooled),
                "value": self.value_head(pooled).squeeze(-1),
                "avail_logits": self.avail_head(pooled),
            }
            if action_index is not None:
                action_emb = self.action_embed(action_index)
                pred_next_latent = self.next_latent_head(
                    torch.cat([pooled, action_emb], dim=-1)
                )
                out["pred_next_latent"] = pred_next_latent
            return out
else:
    ObjectCentricPolicy = None  # type: ignore


# ---------------------- PolicyHelper: load + score ----------------------


class PolicyHelper:
    """Wraps a loaded ObjectCentricPolicy. Provides per-frame scoring + TTT hooks."""

    def __init__(self, model: "ObjectCentricPolicy", device: "torch.device", config: Dict[str, Any]) -> None:
        self.model = model
        self.device = device
        self.config = config
        self.history = int(config.get("history", 4))
        self.max_steps = int(config.get("max_steps", 192))

    @classmethod
    def load(cls, path: Path, device: Optional["torch.device"] = None) -> Optional["PolicyHelper"]:
        """Load a checkpoint produced by `src.train`. Returns None on any failure."""
        if not _TORCH_OK:
            return None
        if device is None:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        try:
            payload = torch.load(str(path), map_location=device, weights_only=False)
        except Exception as exc:
            print(f"[PolicyHelper] checkpoint load failed at {path}: {exc}", flush=True)
            return None
        config = payload.get("config", {}) or {}
        model = ObjectCentricPolicy(
            history=int(config.get("history", 4)),
            model_dim=int(config.get("model_dim", 384)),
            num_slots=int(config.get("num_slots", 8)),
            depth=int(config.get("depth", 6)),
            num_heads=int(config.get("num_heads", 8)),
            scalar_dim=18,
            use_goal=bool(config.get("use_goal", False)),
        )
        try:
            model.load_state_dict(payload["model_state"], strict=False)
        except Exception as exc:
            print(f"[PolicyHelper] state_dict load failed: {exc}", flush=True)
            return None
        model.to(device).eval()
        n_params = sum(p.numel() for p in model.parameters())
        print(
            f"[PolicyHelper] loaded {n_params/1e6:.1f}M-param model from {path} on {device} "
            f"(history={config.get('history')} model_dim={config.get('model_dim')})",
            flush=True,
        )
        return cls(model=model, device=device, config=config)

    @torch.no_grad() if _TORCH_OK else (lambda f: f)
    def score_frame(
        self,
        history_frames: Sequence[Sequence[Sequence[int]]],
        latest_frame: Sequence[Sequence[int]],
        last_action_id: Optional[int],
        levels_completed: int,
        steps_since_progress: int,
        step_index: int,
        available_actions: Sequence[int],
    ) -> Optional[Dict[str, np.ndarray]]:
        """Returns {'action_logits' (7,), 'x_logits' (64,), 'y_logits' (64,),
        'action_scores' softmaxed (7,), 'x_scores' (64,), 'y_scores' (64,),
        'value' scalar} as numpy arrays. Returns None on torch failure.
        """
        if not _TORCH_OK or self.model is None:
            return None
        try:
            obs = frames_to_obs_uint8(history_frames, self.history).unsqueeze(0).to(self.device)
            scalar = scalar_features(
                available_actions=available_actions,
                last_action_id=last_action_id,
                levels_completed=levels_completed,
                steps_since_progress=steps_since_progress,
                step_index=step_index,
                frame=latest_frame,
                max_steps=self.max_steps,
            ).unsqueeze(0).to(self.device)
            out = self.model(obs, scalar)
            action_logits = out["action_logits"].squeeze(0).cpu().numpy()
            x_logits = out["x_logits"].squeeze(0).cpu().numpy()
            y_logits = out["y_logits"].squeeze(0).cpu().numpy()
            value = float(out["value"].squeeze(0).cpu().item())
            return {
                "action_logits": action_logits,
                "x_logits": x_logits,
                "y_logits": y_logits,
                "action_scores": _softmax(action_logits),
                "x_scores": _softmax(x_logits),
                "y_scores": _softmax(y_logits),
                "value": value,
            }
        except Exception as exc:
            print(f"[PolicyHelper] score_frame failed: {exc}", flush=True)
            return None


def _softmax(logits: np.ndarray) -> np.ndarray:
    e = np.exp(logits - logits.max())
    return e / max(1e-8, e.sum())


# ---------------------- TTT: test-time training ----------------------


def ttt_replay_finetune(
    helper: "PolicyHelper",
    replay_actions: List[Dict[str, Any]],
    n_steps: int = 30,
    batch_size: int = 8,
    lr: float = 1e-4,
) -> Dict[str, float]:
    """Adapt the BC checkpoint to a specific game's GT replay.

    `replay_actions` is the list MyAgent already builds from the per-game
    replay JSON (same shape as `_load_replay_actions` returns). Each entry
    is {'type': 'reset'} or {'type': 'action', 'id': int, 'x': int, 'y': int}.

    We don't need (state, action) pairs here — the GT replay only stores
    action sequences, not the resulting frames. So we use a simpler
    objective: train the model's `action_head` to predict the GT action
    from the current observation built from the recent history. Since we
    don't have actual rollout frames, we use a placeholder all-zero history
    and just teach the action distribution. This is a degenerate form of
    TTT — it teaches the model "for this game, prefer this action mix".

    For real per-game replay TTT (with state context), a richer setup
    would replay the game in a sandbox to capture (state, action) pairs.
    This MVP version is the cheapest possible signal.
    """
    if not _TORCH_OK or helper is None or helper.model is None:
        return {"steps": 0, "final_loss": 0.0}
    actions = [int(e.get("id", 0)) for e in replay_actions if e.get("type") == "action"]
    actions = [a for a in actions if 1 <= a <= 7]
    if len(actions) < batch_size:
        return {"steps": 0, "final_loss": 0.0, "skipped": "too_few_actions"}

    helper.model.train()
    opt = torch.optim.AdamW(helper.model.parameters(), lr=lr, weight_decay=0.01)

    # Build the placeholder obs and scalar features once — we're teaching the
    # MARGINAL action distribution for this game.
    blank_history = [[[0 for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)] for _ in range(helper.history)]
    obs_one = frames_to_obs_uint8(blank_history, helper.history).unsqueeze(0).to(helper.device)
    scalar_one = scalar_features(
        available_actions=ACTION_IDS, last_action_id=None,
        levels_completed=0, steps_since_progress=0, step_index=0,
        frame=blank_history[-1], max_steps=helper.max_steps,
    ).unsqueeze(0).to(helper.device)

    action_indices = [ACTION_TO_INDEX[a] for a in actions]
    action_tensor_full = torch.tensor(action_indices, dtype=torch.long, device=helper.device)

    losses: List[float] = []
    rng = np.random.default_rng(0)
    for step in range(n_steps):
        idx = rng.integers(0, len(action_indices), size=batch_size)
        target = action_tensor_full[idx]
        obs_batch = obs_one.expand(batch_size, *obs_one.shape[1:])
        scalar_batch = scalar_one.expand(batch_size, *scalar_one.shape[1:])
        out = helper.model(obs_batch, scalar_batch)
        loss = F.cross_entropy(out["action_logits"], target)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(float(loss.item()))

    helper.model.eval()
    return {"steps": int(n_steps), "final_loss": losses[-1] if losses else 0.0,
            "first_loss": losses[0] if losses else 0.0, "n_actions": len(actions)}


def ttt_aug_consistency(
    helper: "PolicyHelper",
    history_frames: Sequence[Sequence[Sequence[int]]],
    latest_frame: Sequence[Sequence[int]],
    available_actions: Sequence[int],
    last_action_id: Optional[int],
    levels_completed: int,
    steps_since_progress: int,
    step_index: int,
    n_aug: int = 8,
    n_steps: int = 10,
    lr: float = 5e-5,
) -> Dict[str, float]:
    """Augmentation-based TTT for ANY game (works without replays).

    Generate `n_aug` random color-permutation augmentations of the current
    frame. The model's action prediction should be INVARIANT under color
    permutation (color 0 stays as background; colors 1-15 are interchangeable).
    Use the BASE model's prediction on the un-permuted frame as a soft target,
    then KL-distill the augmented predictions toward it.

    This adapts the encoder to be more robust on the current game's specific
    visual content — useful for hidden games where no replay is available.
    """
    if not _TORCH_OK or helper is None or helper.model is None:
        return {"steps": 0}

    # Compute base prediction (no_grad — frozen target).
    helper.model.eval()
    with torch.no_grad():
        obs_base = frames_to_obs_uint8(history_frames, helper.history).unsqueeze(0).to(helper.device)
        scalar_base = scalar_features(
            available_actions=available_actions, last_action_id=last_action_id,
            levels_completed=levels_completed, steps_since_progress=steps_since_progress,
            step_index=step_index, frame=latest_frame, max_steps=helper.max_steps,
        ).unsqueeze(0).to(helper.device)
        base_out = helper.model(obs_base, scalar_base)
        base_action_log_softmax = F.log_softmax(base_out["action_logits"], dim=-1).detach()

    helper.model.train()
    opt = torch.optim.AdamW(helper.model.parameters(), lr=lr, weight_decay=0.01)

    rng = np.random.default_rng(int(time.time() * 1000) & 0xFFFFFFFF)

    losses: List[float] = []
    for step in range(n_steps):
        # Build a batch of n_aug color-permuted versions of the obs.
        permuted_obs_list = []
        for _ in range(n_aug):
            # Random permutation of colors 1..15; color 0 is identity (background).
            rest = list(range(1, NUM_COLORS))
            rng.shuffle(rest)
            perm = np.array([0] + rest, dtype=np.uint8)
            permuted = perm[obs_base.cpu().numpy().astype(np.int64)]
            permuted_obs_list.append(torch.from_numpy(permuted))
        obs_aug = torch.cat(permuted_obs_list, dim=0).to(helper.device)
        scalar_aug = scalar_base.expand(n_aug, *scalar_base.shape[1:])
        aug_out = helper.model(obs_aug, scalar_aug)
        aug_log_softmax = F.log_softmax(aug_out["action_logits"], dim=-1)
        # KL(target || aug) = sum target * (log target - log aug)
        # Use the mean of base_action_log_softmax broadcast across batch.
        target = base_action_log_softmax.exp().expand(n_aug, -1)
        loss = F.kl_div(aug_log_softmax, target, reduction="batchmean")
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(float(loss.item()))

    helper.model.eval()
    return {"steps": int(n_steps), "first_loss": losses[0] if losses else 0.0,
            "final_loss": losses[-1] if losses else 0.0, "n_aug": n_aug}


def ttt_rollout_finetune(
    helper: "PolicyHelper",
    history_buffer: List[List[List[List[int]]]],
    action_buffer: List[Dict[str, Any]],
    next_frame_buffer: List[List[List[int]]],
    goal_frame: Optional[List[List[int]]] = None,
    n_steps: int = 30,
    batch_size: int = 8,
    lr: float = 5e-5,
) -> Dict[str, float]:
    """Real-rollout TTT: adapts the model to the current game's actual dynamics.

    Inputs are buffers collected from the agent's first ~30 actions in this game:
      history_buffer:     list of (history, 64, 64) raw color grids per step
      action_buffer:      list of {'action_id', 'action_data': {'x','y'} or {}}
      next_frame_buffer:  list of (64, 64) raw color grids (post-action frames)
      goal_frame:         optional (64, 64) grid; if provided AND helper.model.use_goal,
                          conditioning is preserved during fine-tuning.

    Loss = action CE + 0.3 * coord CE (when ACTION6) + 0.1 * recon CE (forward model).
    Forward-model loss is the key signal: it adapts the encoder to the
    game's specific visual dynamics, which the policy heads then exploit.

    Returns {'steps', 'first_loss', 'final_loss', 'n_pairs'}.
    """
    if not _TORCH_OK or helper is None or helper.model is None:
        return {"steps": 0}
    n_pairs = min(len(history_buffer), len(action_buffer), len(next_frame_buffer))
    if n_pairs < batch_size:
        return {"steps": 0, "skipped": "too_few_pairs", "n_pairs": n_pairs}

    helper.model.train()
    opt = torch.optim.AdamW(helper.model.parameters(), lr=lr, weight_decay=0.01)

    # Pre-build all (obs, scalar, action_index, next_frame_target) tensors once.
    # Atomic append: build all per-item tensors first, then append all-or-nothing,
    # so a partial failure mid-build doesn't desync obs_list vs next_target_list.
    obs_list, scalar_list, ai_list, x_list, y_list, coord_mask_list, next_target_list = [], [], [], [], [], [], []
    n_drop_skip_aid, n_drop_exc = 0, 0
    last_exc_msg = ""
    for h, ad, nf in zip(history_buffer[:n_pairs], action_buffer[:n_pairs], next_frame_buffer[:n_pairs]):
        try:
            aid = int(ad.get("action_id", 1))
            if aid not in ACTION_TO_INDEX:
                n_drop_skip_aid += 1
                continue
            obs = frames_to_obs_uint8(h, helper.history)
            cur_frame = h[-1] if h else [[0]*GRID_SIZE for _ in range(GRID_SIZE)]
            scalar = scalar_features(
                available_actions=ACTION_IDS, last_action_id=None,
                levels_completed=0, steps_since_progress=0, step_index=0,
                frame=cur_frame, max_steps=helper.max_steps,
            )
            ad_data = ad.get("action_data") or {}
            x_v = int(ad_data.get("x", 0))
            y_v = int(ad_data.get("y", 0))
            cm_v = 1.0 if aid == 6 else 0.0
            next_arr = np.asarray(nf, dtype=np.int64).clip(0, NUM_COLORS - 1)
            if next_arr.shape != (GRID_SIZE, GRID_SIZE):
                pad = np.zeros((GRID_SIZE, GRID_SIZE), dtype=np.int64)
                rh = min(next_arr.shape[0] if next_arr.ndim >= 1 else 0, GRID_SIZE)
                rw = min(next_arr.shape[1] if next_arr.ndim >= 2 else 0, GRID_SIZE)
                if rh and rw:
                    pad[:rh, :rw] = next_arr[:rh, :rw]
                next_arr = pad
            next_t = torch.from_numpy(next_arr)
        except Exception as _exc:
            n_drop_exc += 1
            last_exc_msg = f"{type(_exc).__name__}: {_exc}"
            continue
        # All builds succeeded — atomic append.
        obs_list.append(obs)
        scalar_list.append(scalar)
        ai_list.append(ACTION_TO_INDEX[aid])
        x_list.append(x_v)
        y_list.append(y_v)
        coord_mask_list.append(cm_v)
        next_target_list.append(next_t)

    if len(ai_list) < batch_size:
        helper.model.eval()
        return {
            "steps": 0, "skipped": "build_failed",
            "n_pairs": len(ai_list),
            "n_drop_skip_aid": n_drop_skip_aid,
            "n_drop_exc": n_drop_exc,
            "last_exc_msg": last_exc_msg,
        }

    obs_all = torch.stack(obs_list, dim=0).to(helper.device)
    scalar_all = torch.stack(scalar_list, dim=0).to(helper.device)
    ai_all = torch.tensor(ai_list, dtype=torch.long, device=helper.device)
    x_all = torch.tensor(x_list, dtype=torch.long, device=helper.device)
    y_all = torch.tensor(y_list, dtype=torch.long, device=helper.device)
    coord_mask_all = torch.tensor(coord_mask_list, dtype=torch.float32, device=helper.device)
    next_target_all = torch.stack(next_target_list, dim=0).to(helper.device)

    # Optional goal conditioning.
    use_goal_path = bool(getattr(helper.model, "use_goal", False)) and goal_frame is not None
    if use_goal_path:
        goal_arr = np.asarray(goal_frame, dtype=np.uint8).clip(0, NUM_COLORS - 1)
        if goal_arr.shape != (GRID_SIZE, GRID_SIZE):
            pad = np.zeros((GRID_SIZE, GRID_SIZE), dtype=np.uint8)
            rh = min(goal_arr.shape[0] if goal_arr.ndim >= 1 else 0, GRID_SIZE)
            rw = min(goal_arr.shape[1] if goal_arr.ndim >= 2 else 0, GRID_SIZE)
            if rh and rw:
                pad[:rh, :rw] = goal_arr[:rh, :rw]
            goal_arr = pad
        goal_one = torch.from_numpy(goal_arr).unsqueeze(0).to(helper.device)
    else:
        goal_one = None

    n_data = obs_all.shape[0]
    losses: List[float] = []
    rng = np.random.default_rng(0)
    for step in range(n_steps):
        idx = rng.integers(0, n_data, size=batch_size)
        idx_t = torch.from_numpy(idx).long().to(helper.device)
        ob = obs_all[idx_t]; sc = scalar_all[idx_t]
        ai = ai_all[idx_t]; xt = x_all[idx_t]; yt = y_all[idx_t]
        cm = coord_mask_all[idx_t]; nft = next_target_all[idx_t]
        gob = goal_one.expand(batch_size, *goal_one.shape[1:]) if goal_one is not None else None
        out = helper.model(ob, sc, action_index=ai, goal_obs=gob)
        action_loss = F.cross_entropy(out["action_logits"], ai)
        coord_loss = (
            F.cross_entropy(out["x_logits"], xt, reduction="none") +
            F.cross_entropy(out["y_logits"], yt, reduction="none")
        )
        coord_loss = (coord_loss * cm).mean() * 0.5
        recon_loss = torch.tensor(0.0, device=helper.device)
        if "next_frame_recon_logits" in out:
            # next_frame_recon_logits: (B, NUM_COLORS, 64, 64)
            recon_loss = F.cross_entropy(out["next_frame_recon_logits"], nft) * 0.1
        loss = action_loss + 0.3 * coord_loss + recon_loss
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(float(loss.item()))

    helper.model.eval()
    return {
        "steps": int(n_steps),
        "first_loss": losses[0] if losses else 0.0,
        "final_loss": losses[-1] if losses else 0.0,
        "n_pairs": int(n_data),
    }


def find_checkpoint() -> Optional[Path]:
    """Search canonical paths for a BC checkpoint. Same idea as the dict loader."""
    candidates = [
        os.environ.get("ARC_BC_CHECKPOINT_PATH"),
        "/kaggle/working/best.pth",
        "/kaggle/input/arc-agi-3-replays-v1/best.pth",
        # Local dev paths.
        "/mnt/c/Users/ljh20/MCS/ARC-Prize-2026-ARC-AGI-3/Training_Output/bc_v2_filtered_local_v1/checkpoints/best.pth",
    ]
    for p in candidates:
        if not p:
            continue
        path = Path(p)
        if path.is_file():
            return path
    return None


In [ ]:
# --- Cell 5: in rerun mode, set up the framework and run main.py --- #
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the gateway HTTP service to be ready
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy the framework to writable location
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Drop our agent into the framework's templates folder
    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Minimal agents/__init__.py — original eagerly imports llm / langgraph
    # templates whose deps aren't installed.
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    'random': Random,
    'myagent': MyAgent,
}
""")

    # .env points the framework at the gateway.
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
ARC_REPLAY_BASE_DIR=/kaggle/working/replays
""")

    # Run the agent. main.py iterates gateway's games and invokes MyAgent.
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        ARC_GOOSE_DELTA=t1_bc \
        ARC_BC_CHECKPOINT_PATH=/kaggle/working/best.pth \
        ARC_REPLAY_BASE_DIR=/kaggle/working/replays \
        python main.py --agent myagent

In [ ]:
# --- Cell 6: in dev mode, write dummy submission.parquet --- #
# The grader replaces this with real scoring during rerun.
import os

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print('dummy submission.parquet written (dev mode)')